In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

df_model = pd.read_csv("data/df_model_clean.csv")
df_clean = pd.read_csv("data/df_clean.csv")

In [2]:
from sklearn.model_selection import train_test_split

X = df_model.drop(columns=['two_year_recid'])
y = df_model['two_year_recid']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (4937, 10), Test: (1235, 10)


In [3]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)
print(f"Train accuracy: {train_acc:.3f}")
print(f"Test accuracy: {test_acc:.3f}")

Train accuracy: 0.674
Test accuracy: 0.687


In [4]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.68      0.79      0.73       673
           1       0.69      0.57      0.62       562

    accuracy                           0.69      1235
   macro avg       0.69      0.68      0.68      1235
weighted avg       0.69      0.69      0.68      1235

[[530 143]
 [244 318]]


In [5]:
#Inspecting coefficients; to see what the model has learned.
coefficients = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)

print(coefficients)

                 feature  coefficient
4  race_African-American     0.310815
6         race_Caucasian     0.260126
9             race_Other     0.161826
2           priors_count     0.160126
8   race_Native American     0.059329
7          race_Hispanic    -0.033557
1                    age    -0.044407
5             race_Asian    -0.190430
3        c_charge_degree    -0.303514
0                    sex    -0.379152


In [6]:
X_train_ref = X_train.drop(columns=['race_Caucasian'])
X_test_ref = X_test.drop(columns=['race_Caucasian'])

model_ref = LogisticRegression(max_iter=1000)
model_ref.fit(X_train_ref, y_train)

coefficients_ref = pd.DataFrame({
    'feature': X_train_ref.columns,
    'coefficient': model_ref.coef_[0]
}).sort_values('coefficient', ascending=False)

print(coefficients_ref)
print(f"Test accuracy: {model_ref.score(X_test_ref, y_test):.3f}")

                 feature  coefficient
2           priors_count     0.160048
4  race_African-American     0.053134
1                    age    -0.044514
7   race_Native American    -0.055632
8             race_Other    -0.094819
6          race_Hispanic    -0.290295
3        c_charge_degree    -0.305123
0                    sex    -0.379554
5             race_Asian    -0.396120
Test accuracy: 0.686


In [9]:
df_clean.groupby('race')['priors_count'].mean()

race
African-American    4.238110
Asian               1.354839
Caucasian           2.289111
Hispanic            2.100196
Native American     5.181818
Other               1.746356
Name: priors_count, dtype: float64

In [10]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)

train_acc_rf = rf_model.score(X_train, y_train)
test_acc_rf = rf_model.score(X_test, y_test)
print(f"Train accuracy: {train_acc_rf:.3f}")
print(f"Test accuracy: {test_acc_rf:.3f}")

Train accuracy: 0.701
Test accuracy: 0.702


In [11]:
y_pred_rf = rf_model.predict(X_test)
print(classification_report(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.70      0.78      0.74       673
           1       0.70      0.61      0.65       562

    accuracy                           0.70      1235
   macro avg       0.70      0.69      0.70      1235
weighted avg       0.70      0.70      0.70      1235

[[525 148]
 [220 342]]


In [12]:
from sklearn.metrics import roc_auc_score

y_proba_lr = model.predict_proba(X_test)[:, 1]
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print(f"Logistic Regression ROC-AUC: {roc_auc_score(y_test, y_proba_lr):.3f}")
print(f"Random Forest ROC-AUC: {roc_auc_score(y_test, y_proba_rf):.3f}")

Logistic Regression ROC-AUC: 0.729
Random Forest ROC-AUC: 0.735


In [13]:
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(importances)

                 feature  importance
2           priors_count    0.517783
1                    age    0.312192
3        c_charge_degree    0.052956
4  race_African-American    0.042586
0                    sex    0.042166
6         race_Caucasian    0.012500
7          race_Hispanic    0.008148
9             race_Other    0.005500
5             race_Asian    0.004098
8   race_Native American    0.002072


In [15]:
import joblib
import os
os.makedirs("models", exist_ok=True)
joblib.dump(rf_model, "models/rf_model.pkl")

['models/rf_model.pkl']

In [16]:
X_test.to_csv("data/X_test.csv", index=True)
y_test.to_csv("data/y_test.csv", index=True)